[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/00_onboarding/00e_git_and_github_basics.ipynb)

# 🌿 Notebook 0e — Git & GitHub (the parts you actually use)

> **Module:** Onboarding · **Estimated time:** 45–55 min · **Difficulty:** Beginner · **Prerequisites:** [Notebook 0d — the command line](00d_command_line_basics.ipynb)

**Git** is a time machine for your code. It records **snapshots** (called *commits*) so you can see what changed, when, and why — and roll back when today's "improvement" turns out to break everything. **GitHub** is a website that hosts git repositories so you can back them up, share them, and collaborate.

Git has a fearsome reputation, mostly because it has ~150 commands and every tutorial shows you 40 of them. You need about **eight**: `init`, `status`, `add`, `commit`, `log`, `diff`, `branch`/`switch`, and `merge` — plus two undo commands and the GitHub verbs `clone`/`push`/`pull`. This lesson runs each one for real.

## 🎯 Learning objectives

By the end you can:

1. Explain the three "places" a file lives in git — **working directory**, **staging area**, **repository** — the mental model everything else hangs off.
2. Start a repo and record snapshots — `git init`, `add`, `commit`, and read history with `log` and `diff`.
3. Work on a **branch** and **merge** it back — the everyday unit of "a change in progress."
4. **Undo** safely — discard a working change, unstage, and understand what `reset` does.
5. Keep junk out of history with **`.gitignore`**.
6. Explain what **GitHub** adds — `clone`, `remote`, `push`, `pull`, pull requests, and forks — and why the course links every notebook to it.

> 🧭 **How this notebook stays runnable & offline.** Git is fully local — `init/add/commit/branch/merge` need no network — so every git example below runs **for real** in a throwaway repo under `/tmp`, and you see genuine output. Only *GitHub* needs an account and a network, so those steps are **described**, not executed (exactly how Modules 9 & 14 handle it). Copy any command into a real repo and it behaves identically.

## 1. The one mental model: three places a change lives

This is the idea that makes git click. A change moves through **three areas**:

```
  working directory   →   staging area   →   repository
   (your edits,          (git add:         (git commit:
    not yet tracked)      "include this     "save this snapshot
                          in the next        for good")
                          snapshot")
```

- **Working directory** — the actual files on disk, as you're editing them.
- **Staging area** (a.k.a. the *index*) — a holding pen. You `git add` the specific changes you want in the *next* snapshot. This lets you commit *some* of your edits and leave others for later.
- **Repository** — the permanent history. `git commit` freezes whatever is staged into a new snapshot with a message.

Beginners forget the middle box and wonder why `commit` "did nothing" — the answer is almost always *"you didn't `add` it first."* Keep this diagram in your head for the rest of the lesson.

In [1]:
import subprocess, tempfile, os

REPO = tempfile.mkdtemp(prefix="git_lesson_")

def git(command, cwd=REPO, show=True):
    """Run a shell command in the throwaway repo and print it like a terminal."""
    r = subprocess.run(command, shell=True, cwd=cwd, capture_output=True,
                        text=True, executable="/bin/bash")
    if show:
        print(f"$ {command}")
        out = (r.stdout + r.stderr).rstrip()
        if out:
            print(out)
    return r

print("Throwaway repo folder:", REPO)

Throwaway repo folder: /var/folders/sz/1k1y5gg975j3mc23vxwrt0v40000gn/T/git_lesson_gxcc_gm1


## 2. Starting a repo — `git init` and a one-time identity

- `git init -b main` — turn the current folder into a git repository, with the first branch named **main** (`-b main` makes it deterministic; older git defaults to `master`). This creates a hidden `.git/` folder — that folder *is* the repository (the whole history lives there).
- `git config user.name` / `user.email` — git stamps every commit with *who made it*. You normally set this **once globally** (`git config --global user.name "Your Name"`). Here we set it **locally** (just for this throwaway repo) so the lesson is self-contained and doesn't touch your real settings.

In [2]:
git("git init -b main")
# Local identity — real life: run these once with --global instead of --local.
git("git config --local user.name  'Course Learner'")
git("git config --local user.email 'learner@example.com'")
print()
git("ls -a")            # note the new .git/ folder — that's the repository itself

$ git init -b main
Initialized empty Git repository in /private/var/folders/sz/1k1y5gg975j3mc23vxwrt0v40000gn/T/git_lesson_gxcc_gm1/.git/
$ git config --local user.name  'Course Learner'
$ git config --local user.email 'learner@example.com'

$ ls -a
.
..
.git


CompletedProcess(args='ls -a', returncode=0, stdout='\x1b.\x1b[m\x1b[m\n\x1b..\x1b[m\x1b[m\n\x1b.git\x1b[m\x1b[m\n', stderr='')

## 3. Your first snapshot — `status`, `add`, `commit`

The daily loop is: **edit → `status` → `add` → `commit`.**

- `git status` — the command you run more than any other: *what has changed, and what's staged?* When you're lost, run `status`.
- `git add FILE` (or `git add .` for everything) — move changes into the staging area.
- `git commit -m "message"` — freeze the staged changes into a snapshot. The message says *why*; write it for the teammate (or future you) who'll read it in six months.

In [3]:
# Create two files in the working directory
git('echo "print(\'hello\')" > app.py')
git('echo "# My Project" > README.md')
print()
git("git status")               # both show as "Untracked" — git sees them but isn't tracking them yet

$ echo "print('hello')" > app.py
$ echo "# My Project" > README.md

$ git status
On branch main

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	README.md
	app.py

nothing added to commit but untracked files present (use "git add" to track)


CompletedProcess(args='git status', returncode=0, stdout='On branch main\n\nNo commits yet\n\nUntracked files:\n  (use "git add <file>..." to include in what will be committed)\n\tREADME.md\n\tapp.py\n\nnothing added to commit but untracked files present (use "git add" to track)\n', stderr='')

In [4]:
git("git add app.py README.md")   # stage both
print()
git("git status")                 # now "Changes to be committed" (staged, green in a real terminal)
print()
git('git commit -m "Initial commit: app and README"')
print()
git("git status")                 # "nothing to commit, working tree clean" — the goal state

$ git add app.py README.md

$ git status
On branch main

No commits yet

Changes to be committed:
  (use "git rm --cached <file>..." to unstage)
	new file:   README.md
	new file:   app.py

$ git commit -m "Initial commit: app and README"
[main (root-commit) ab07dc2] Initial commit: app and README
 2 files changed, 2 insertions(+)
 create mode 100644 README.md
 create mode 100644 app.py

$ git status
On branch main
nothing to commit, working tree clean


CompletedProcess(args='git status', returncode=0, stdout='On branch main\nnothing to commit, working tree clean\n', stderr='')

## 4. Reading history — `log` and `diff`

- `git log` — the list of commits, newest first, each with its author, date, 40-character **hash** (its unique id), and message. `git log --oneline` is the compact version you'll use most.
- `git diff` — show *unstaged* changes line by line (`+` added, `-` removed). `git diff --staged` shows what you've `add`ed but not yet committed. This is how you review your own work *before* committing it — a habit worth building early.

In [5]:
# Make an edit, then inspect it before committing
git('echo "print(\'goodbye\')" >> app.py')   # append a line to app.py
print()
git("git diff")                                # exactly what changed, line by line
print()
git("git add app.py")
git('git commit -m "Add a goodbye line"')
print()
git("git log --oneline")                       # two commits now, compact view

$ echo "print('goodbye')" >> app.py

$ git diff
diff --git a/app.py b/app.py
index b376c99..d94cc99 100644
--- a/app.py
+++ b/app.py
@@ -1 +1,2 @@
 print('hello')
+print('goodbye')

$ git add app.py
$ git commit -m "Add a goodbye line"
[main 39ac18c] Add a goodbye line
 1 file changed, 1 insertion(+)

$ git log --oneline
39ac18c Add a goodbye line
ab07dc2 Initial commit: app and README


CompletedProcess(args='git log --oneline', returncode=0, stdout='39ac18c Add a goodbye line\nab07dc2 Initial commit: app and README\n', stderr='')

> 🔬 **The hash.** Each commit's id (e.g. `a1b2c3d`) is a fingerprint of its content and its parent, so history is tamper-evident and every snapshot is addressable — `git show a1b2c3d` reprints any past commit. You'll see short 7-character hashes everywhere; they're just the front of the full 40.

## 5. Branches — work without fear — `switch` / `branch` / `merge`

A **branch** is a movable label pointing at a commit — a parallel line of work. You make a branch to try something (a feature, a fix, an experiment) *without* disturbing `main`; if it works, you **merge** it back; if it doesn't, you throw the branch away and `main` never knew.

- `git switch -c NAME` — create branch NAME and move onto it (`-c` = create). (Older git: `git checkout -b NAME`.)
- `git branch` — list branches; `*` marks the one you're on.
- `git switch main` — hop back.
- `git merge NAME` — fold NAME's commits into your current branch.

This is the single most important git habit: **`main` stays working; new work happens on a branch.** It's exactly the workflow Module 14 (CI/CD) automates.

In [6]:
git("git switch -c add-tests")          # branch off main to add a test
git('echo "def test_ok(): assert True" > test_app.py')
git("git add test_app.py")
git('git commit -m "Add a first test"')
print()
git("git branch")                       # two branches; * = current (add-tests)
print()
git("git switch main")                  # go back to main...
git("ls")                               # ...and note: test_app.py is NOT here — it lives on the branch

$ git switch -c add-tests
Switched to a new branch 'add-tests'
$ echo "def test_ok(): assert True" > test_app.py


$ git add test_app.py
$ git commit -m "Add a first test"
[add-tests 2c4f18c] Add a first test
 1 file changed, 1 insertion(+)
 create mode 100644 test_app.py

$ git branch
* add-tests
  main



$ git switch main
Switched to branch 'main'
$ ls
README.md
app.py


CompletedProcess(args='ls', returncode=0, stdout='README.md\napp.py\n', stderr='')

In [7]:
git("git merge add-tests")              # fold the branch's work into main
print()
git("ls")                               # now test_app.py is here
print()
git("git log --oneline")                # main's history now includes the test commit

$ git merge add-tests
Updating 39ac18c..2c4f18c
Fast-forward
 test_app.py | 1 +
 1 file changed, 1 insertion(+)
 create mode 100644 test_app.py

$ ls
README.md
app.py
test_app.py



$ git log --oneline
2c4f18c Add a first test
39ac18c Add a goodbye line
ab07dc2 Initial commit: app and README


CompletedProcess(args='git log --oneline', returncode=0, stdout='2c4f18c Add a first test\n39ac18c Add a goodbye line\nab07dc2 Initial commit: app and README\n', stderr='')

## 6. Keeping junk out — `.gitignore`

You do **not** want secrets, huge data files, virtual environments, or generated caches in your history. A file named **`.gitignore`** lists patterns git should pretend it can't see — they'll never show up in `status` or get added by `git add .`.

Typical entries for a Python/AI project:

```
.venv/            # virtual environment — recreate it, don't commit it
__pycache__/      # compiled bytecode
*.pkl             # model files — often huge
.env              # SECRETS: API keys live here, never in git
data/*.csv        # large/regenerated data
```

> ⚠️ **The most expensive git mistake is committing a secret.** Once an API key is pushed to GitHub it's effectively public forever (it's in history even after you delete it) — you must rotate it. `.gitignore` your `.env` *before* your first commit.

In [8]:
# Without .gitignore, a stray .env and cache would show up as untracked:
git('echo "OPENAI_API_KEY=sk-secret" > .env')
git("mkdir __pycache__ && touch __pycache__/app.cpython.pyc")
git("git status --short")               # git currently sees .env and the cache
print()
# Add a .gitignore and watch them vanish from git's view:
git('printf ".env\n__pycache__/\n" > .gitignore')
git("git status --short")               # now only .gitignore is untracked; .env is ignored
print()
git("git add .gitignore && git commit -m 'Add .gitignore (ignore secrets & caches)'")

$ echo "OPENAI_API_KEY=sk-secret" > .env
$ mkdir __pycache__ && touch __pycache__/app.cpython.pyc
$ git status --short
?? .env
?? __pycache__/



$ printf ".env
__pycache__/
" > .gitignore
$ git status --short
?? .gitignore

$ git add .gitignore && git commit -m 'Add .gitignore (ignore secrets & caches)'
[main 3ff587f] Add .gitignore (ignore secrets & caches)
 1 file changed, 2 insertions(+)
 create mode 100644 .gitignore


CompletedProcess(args="git add .gitignore && git commit -m 'Add .gitignore (ignore secrets & caches)'", returncode=0, stdout='[main 3ff587f] Add .gitignore (ignore secrets & caches)\n 1 file changed, 2 insertions(+)\n create mode 100644 .gitignore\n', stderr='')

## 7. Undoing things — the three you'll actually reach for

Git's safety net. Match the tool to *where* the change is (remember the three areas):

- **Discard an unstaged edit** (working dir): `git restore FILE` — throw away changes since the last commit. *(Gone for good — like `rm`, no undo.)*
- **Unstage** (staging → working dir): `git restore --staged FILE` — you `add`ed something by mistake; this takes it back out of the snapshot without losing the edit.
- **Move the branch back** (repository): `git reset --soft HEAD~1` undoes the last *commit* but keeps the changes staged; `git reset --hard HEAD~1` undoes the commit **and** discards the changes (dangerous).

`HEAD` means "the commit I'm currently on"; `HEAD~1` is "one before that."

In [9]:
# Stage something, then change your mind
git('echo "junk" > scratch.txt')
git("git add scratch.txt")
git("git status --short")               # A = added/staged
print()
git("git restore --staged scratch.txt") # unstage it (edit stays on disk)
git("git status --short")               # ?? = untracked again
print()
git("rm scratch.txt")                   # here we just delete the junk file
git("git status --short")               # clean

$ echo "junk" > scratch.txt


$ git add scratch.txt
$ git status --short
A  scratch.txt

$ git restore --staged scratch.txt
$ git status --short
?? scratch.txt

$ rm scratch.txt
$ git status --short


CompletedProcess(args='git status --short', returncode=0, stdout='', stderr='')

## 8. GitHub — git's home in the cloud

Everything so far was **local** — the repo lives only on your machine. **GitHub** (and GitLab, Bitbucket) hosts a copy online: a backup, a share link, and the place teams collaborate. This is what the *Open in Colab* badge at the top of every course notebook points to.

A **remote** is a named pointer to that online copy — conventionally called `origin`. The one command below is safe to run (it just records a URL; nothing is sent):

- `git remote add origin URL` — link your local repo to a GitHub repo.
- `git remote -v` — list your remotes.

The commands that actually talk to the network (they need an account + login, so they're **described, not run** here):

| Command | What it does |
|---|---|
| `git clone URL` | Download a full repo (history and all) to your machine — how you *start* from someone else's project. |
| `git push` | Upload your local commits to the remote (`git push -u origin main` the first time). |
| `git pull` | Download and merge commits others pushed — do this before you start work each day. |

**Pull requests (PRs)** are a GitHub feature, not a git command: you push a branch, open a PR on the website, and teammates review and discuss before it merges into `main`. **Forking** makes your own copy of someone else's repo so you can propose changes without write access — how open-source contribution works.

In [10]:
# Safe & local: naming a remote just stores a URL — no network, nothing uploaded.
git("git remote add origin https://github.com/your-name/your-repo.git")
git("git remote -v")
print()
print("In a real project you'd now run:  git push -u origin main")
print("(needs a GitHub account + login, so we stop here.)")

$ git remote add origin https://github.com/your-name/your-repo.git


$ git remote -v
origin	https://github.com/your-name/your-repo.git (fetch)
origin	https://github.com/your-name/your-repo.git (push)

In a real project you'd now run:  git push -u origin main
(needs a GitHub account + login, so we stop here.)


## 🧪 Practice exercises

Run these in a throwaway folder in your **own** terminal (`mkdir /tmp/gitplay && cd /tmp/gitplay`). Solutions below.

### Exercise 1 — ⭐ The core loop
Initialise a repo, create `hello.txt` containing `hi`, and make your first commit. Then show the one-line log.

### Exercise 2 — ⭐⭐ Branch, change, merge
On a new branch `edit`, change `hello.txt` to say `hello world`, commit it, switch back to `main`, and merge `edit` in. Confirm `main` now has the new content.

### Exercise 3 — ⭐⭐ Ignore a secret
Add a `.env` file with a fake key, then add a `.gitignore` so `git status` no longer shows it.

<details>
<summary>💡 <b>Solutions</b></summary>

```bash
# Exercise 1
git init -b main
echo "hi" > hello.txt
git add hello.txt
git commit -m "First commit"
git log --oneline

# Exercise 2
git switch -c edit
echo "hello world" > hello.txt
git add hello.txt
git commit -m "Update greeting"
git switch main
git merge edit
cat hello.txt          # → hello world

# Exercise 3
echo "API_KEY=sk-fake" > .env
git status --short     # shows .env  (bad!)
echo ".env" > .gitignore
git status --short     # .env is gone; only .gitignore shows
```

The whole of everyday git is in Exercise 2: **branch → change → commit → switch → merge.** Everything else is variations on it.
</details>

## ✅ Self-assessment

- [ ] I can draw the three areas — working directory, staging, repository — and say what `add` and `commit` move between them.
- [ ] `git init`, `status`, `add`, `commit` — I can start a repo and record a snapshot.
- [ ] `git log --oneline` and `git diff` — I can read history and review a change before committing.
- [ ] I can create a branch, commit on it, and `merge` it back into `main`.
- [ ] I can discard an edit, unstage a file, and I know `reset --hard` is the dangerous one.
- [ ] I have a `.gitignore` reflex for `.env`, `.venv/`, caches, and big data.
- [ ] I can explain `clone` / `push` / `pull` and what a pull request is.

## 🧠 Key takeaways

1. **Three areas, two verbs:** edits sit in the *working directory*, `git add` stages them, `git commit` saves the snapshot. Forgetting to `add` is the #1 beginner confusion.
2. **`main` stays working; new work happens on a branch** — then `merge`. This one habit prevents most disasters and is what CI/CD automates.
3. **Commit messages and `git diff` are for humans** — your future self and your reviewers. Small, well-described commits beat one giant "stuff" commit.
4. **`.gitignore` before your first commit** — never let a secret reach GitHub; if one does, rotate it immediately.
5. **GitHub = git + a network:** `clone` to start, `push`/`pull` to sync, pull requests to collaborate.

## 🚀 Next step

You now have the two tools every later notebook assumes — a terminal and version control. Head into **[Module 1 — Foundations](../01_foundations/01_python_basics.ipynb)** and start writing Python, or use the [course overview](00b_course_overview.ipynb) to pick a learning path. When a lesson later says *"commit your work"* or *"open a terminal and run…"*, you'll know exactly what to do.

In [11]:
import shutil
shutil.rmtree(REPO, ignore_errors=True)
print("Throwaway repo removed. That's the whole of everyday git — you're set. 🌿")

Throwaway repo removed. That's the whole of everyday git — you're set. 🌿
